<a href="https://colab.research.google.com/github/svirfneblyn2/genAIcource/blob/main/notebooks/L05_Video_Generation_API_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 05: Video Generation — Your First Veo Clip from Code
**Course:** GenAI Basics | **Instructor:** Igor Rubanovich (EPAM Systems) | **Model:** Veo 3.1 Lite / Fast (Gemini API)

This notebook is **optional**. Everything from the lesson also works in the **Gemini app**, **Flow** or **[AI Studio](https://aistudio.google.com)** without code. Here you see the same thing through the API: send a prompt, wait, download the MP4.

| Step | What happens |
|---|---|
| 0 | Connect: install the SDK, read your API key |
| 1 | Text → video: one short prompt, 4 seconds |
| 2 | Image → video: animate the lesson keyframe with the 6-slot director prompt |
| 3 | What can go wrong (and what is normal) |

**How video generation works in an API:** unlike text, a video is not returned right away. You start a *job*, the job runs for 1–6 minutes, and you check on it every few seconds (*polling*). When it is done, you download the file.

> 💳 **Every generation costs money.** Veo has **no free tier** in the Gemini API: you need a key with billing enabled. You pay per second of video, and only for videos that were generated successfully.
>
> | Model (720p) | Price per second | 4 s clip | 8 s clip |
> |---|---|---|---|
> | **Veo 3.1 Lite** (default here) | $0.05 | **$0.20** | $0.40 |
> | Veo 3.1 Fast | $0.10 | $0.40 | $0.80 |
> | Veo 3.1 (standard) | $0.40 | $1.60 | $3.20 |
>
> Prices from [ai.google.dev/gemini-api/docs/pricing](https://ai.google.dev/gemini-api/docs/pricing), October 2026 — check before you run. The paid cells only work after you set `RUN_PAID = True`, so **"Run all" never spends money**.

## Step 0: Connect (same key as L03–L04)
1. Create a key at **[aistudio.google.com/apikey](https://aistudio.google.com/apikey)** and turn on billing for its project (video needs it).
2. In Colab: 🔑 **Secrets** → **+ Add new secret** → name `GEMINI_API_KEY`, paste the key, switch on **Notebook access**.
   Outside Colab: set the environment variable `GEMINI_API_KEY`.
3. Run the cell below. It also defines one helper, `make_video()`, used by both steps: start the job → wait → save the MP4 → show it.

In [ ]:
!pip install -q -U google-genai

import os, time, pathlib
from google import genai
from google.genai import types
from IPython.display import Video, display

# --- API key: Colab secret first, then the environment variable ---
try:
    from google.colab import userdata
    API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:                       # not in Colab, or the secret is missing
    API_KEY = os.environ.get("GEMINI_API_KEY")
if not API_KEY:
    raise RuntimeError("No API key. Add the GEMINI_API_KEY secret in Colab or set the environment variable.")

client = genai.Client(api_key=API_KEY)

# --- Safety switch: paid cells do nothing until you change this to True ---
RUN_PAID = False

# Cheapest model first. Other options: "veo-3.1-fast-generate-preview", "veo-3.1-generate-preview"
MODEL = "veo-3.1-lite-generate-preview"
PRICE_PER_SECOND = {"veo-3.1-lite-generate-preview": 0.05,    # USD, 720p
                    "veo-3.1-fast-generate-preview": 0.10,
                    "veo-3.1-generate-preview": 0.40}

OUT = pathlib.Path("videos_out"); OUT.mkdir(exist_ok=True)


def make_video(name, prompt, image=None, seconds=4, aspect="16:9"):
    """Prompt (+ optional first-frame image) -> MP4 file in videos_out/. Returns the path."""
    cost = PRICE_PER_SECOND.get(MODEL, 0) * seconds
    if not RUN_PAID:
        print(f"Skipped (RUN_PAID = False). This call would cost about ${cost:.2f} ({MODEL}, {seconds} s).")
        return None

    print(f"Starting job: {MODEL}, {seconds} s, {aspect}, ~${cost:.2f}")
    operation = client.models.generate_videos(
        model=MODEL,
        prompt=prompt,
        image=image,                                  # None = text-to-video; an image = its first frame
        config=types.GenerateVideosConfig(
            duration_seconds=seconds,                 # Veo 3.1: 4, 6 or 8
            aspect_ratio=aspect,                      # "16:9" or "9:16"
            resolution="720p",                        # cheapest; 1080p/4k need 8 s
            number_of_videos=1,
        ),
    )

    # Polling: the job runs on Google's side, we just ask "done yet?" every 10 seconds
    start = time.time()
    while not operation.done:
        print(f"  ...still generating ({int(time.time() - start)} s)")
        time.sleep(10)
        operation = client.operations.get(operation)

    if operation.error:
        raise RuntimeError(f"Job failed: {operation.error}")
    result = operation.response
    if not result or not result.generated_videos:
        reasons = getattr(result, "rai_media_filtered_reasons", None)
        raise RuntimeError(f"No video returned (blocked by safety filters?). Reasons: {reasons}")

    video = result.generated_videos[0]
    client.files.download(file=video.video)          # fetch the bytes from Google's storage
    path = OUT / f"{name}.mp4"
    video.video.save(str(path))
    print(f"Done in {int(time.time() - start)} s -> saved {path}")
    display(Video(str(path), embed=True, width=640))
    return path


print(f"Client ready. Model: {MODEL}. RUN_PAID = {RUN_PAID}")

---
## Step 1: Text → video (the simplest call)
A short prompt, the cheapest model, the shortest clip. Veo adds **sound automatically** (ambience, effects): unmute the player.

To actually generate: set `RUN_PAID = True` in Step 0, run that cell again, then run this one. Expect **1–6 minutes** of waiting.

In [ ]:
T2V_PROMPT = (
    "A cozy coffee shop in the early morning, steam rising slowly from a ceramic mug on a wooden counter, "
    "warm sunlight through the window, slow push-in camera move, realistic, calm ambient cafe sounds."
)
make_video("01_t2v_coffee", T2V_PROMPT, seconds=4)

**Look at:** what did the model decide on its own (people? camera speed? what sounds?). Every detail you did not write is a coin toss — that is why the next step starts from a picture.

---
## Step 2: Image → video with the director prompt
The lesson rule: **the picture fixes *who* and *where*; the prompt describes only *what changes*** — movement, camera, light.

We use the course keyframe `shot1_cafe_pickup.png` (a courier picking up an order at a cafe counter). The cell below loads it from GitHub; if that fails, it looks for a local copy, and in Colab it asks you to upload any picture (yours works too — then edit the prompt to match it).

In [ ]:
import urllib.request

KEYFRAME_URL = ("https://raw.githubusercontent.com/svirfneblyn2/genAIcource/main/"
                "GenAI%20Basics%20%E2%80%94%20Course%20Production/L05/demo_veo_showcase/shot1_cafe_pickup.png")
LOCAL_COPY = pathlib.Path("shot1_cafe_pickup.png")

if not LOCAL_COPY.exists():
    try:
        urllib.request.urlretrieve(KEYFRAME_URL, LOCAL_COPY)
        print("Keyframe downloaded from GitHub.")
    except Exception as e:
        print(f"Could not download the keyframe ({e}).")
        try:
            from google.colab import files
            print("Upload a PNG or JPG to use as the first frame:")
            uploaded = files.upload()
            LOCAL_COPY = pathlib.Path(next(iter(uploaded)))
        except ImportError:
            raise RuntimeError("Put an image next to this notebook and set LOCAL_COPY to its file name.")

keyframe = types.Image.from_file(location=str(LOCAL_COPY))   # Veo takes it as the first frame

from PIL import Image as PILImage
preview = PILImage.open(LOCAL_COPY); preview.thumbnail((520, 520)); display(preview)

**The 6-slot director prompt** — the same six slots, in the same order, as on lesson slide 8. Write every slot as *what should happen*, not as "no ...": the model keeps what you describe, and a "no X" line can put X into its head. Do not retell the picture: it already shows who and where, so one short identifying detail is enough.

| # | Slot | Question it answers |
|---|---|---|
| 1 | Camera | How does the camera move? One move, at a named speed |
| 2 | Who | Who moves? One short detail from the picture so they stay the same |
| 3 | Action | What exactly happens in 4 seconds? One clear action, active verbs |
| 4 | Where and when | Place and time of day — short, the picture already shows it |
| 5 | Motion details | What else moves and how: hands, bag, hair, steam, light flicker |
| 6 | Light | Light direction and mood — keep it as in the picture |

In [ ]:
DIRECTOR_PROMPT = """
Steady medium shot at eye level with a slow gentle push-in.
The same courier with short curly hair and a yellow jacket
takes the brown paper bag from the barista's hand with a smile, holds it close with both hands and turns toward the exit in one smooth movement
at the wooden pickup counter of a small cafe on a sunny morning.
Her fingers wrap firmly around the paper bag, the bag keeps its shape, soft dust drifts slowly in the sunbeam.
Warm natural window light from the left, matching the image.
""".strip()

make_video("02_i2v_cafe_pickup", DIRECTOR_PROMPT, image=keyframe, seconds=4)

**Compare with Step 1:** the face, clothes and room come from the picture, so only the movement is new. If something drifts, change **one slot** and run again — not the whole prompt.

Want a sharper or longer clip? `seconds=8` doubles the price; `MODEL = "veo-3.1-fast-generate-preview"` doubles it again. 1080p needs an 8-second clip; 4K needs an 8-second clip **and** Fast or Standard (Lite has no 4K).

---
## Step 3: What can go wrong (and what is normal)
| What you see | Why | What to try |
|---|---|---|
| Extra or melting fingers, a hand passing *through* the bag | Contacts between hands and objects are the hardest thing for video models | Shorter, simpler action; describe the grip; start from a frame where the hand already holds the object |
| Garbled signs, labels, screens | Video models are weak at text, and letters drift from frame to frame | Keep text out of the shot or add it later in an editor |
| Face or clothes slowly "morph" | Each frame is predicted from noise; small errors add up | Image → video, describe the character as in the picture, 4–6 s clips instead of 8 |
| Same prompt, different result | Generation is random. Veo has a `seed` setting, but Google's docs say it "doesn't guarantee determinism" | Save prompt + image + model in a log; judge *what was preserved*, not exact pixels |
| Job finished but no video | Safety filters blocked the prompt or the image (you are not charged) | Rephrase; avoid real people's names and brands |
| `429` / quota / billing error | No billing on the key, or rate limit | Enable billing, wait a minute, or use the Gemini app / Flow instead |

**Watermark:** every Veo video carries an invisible **SynthID** watermark. You cannot see it, it survives compression and cropping, and Google tools (for example the Gemini app) can detect it. This is part of AI-content labelling, not a bug.

**Also worth knowing:** Google now recommends **Gemini Omni Flash** as the default video model in the Gemini API (longer clips, editing by conversation). Veo 3.1 stays available and is the simplest way to learn the "job → wait → download" pattern shown here.